# **ViT(Vision Transformer)**

## 1.환경준비

### (1)라이브러리 로딩

In [1]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, ViTForImageClassification

import os
import numpy as np
from keras.datasets import cifar10, fashion_mnist
from sklearn.metrics import *
from tqdm.auto import tqdm

# 공통: 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\USER\anaconda3\envs\aivle_pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### (2) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (3) OpenAI API Key 등록
* 환경변수로 key 등록

In [3]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '../'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'API_KEY.env')

* ⚠️ 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [4]:
print(os.environ['OPENAI_API_KEY'][:3])

sk-


## 2.ViT 사용해보기

### (1) ViT 기본 모델 다운로드

In [6]:
# 모델 다운로드
model_id = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_id)
model1 = ViTForImageClassification.from_pretrained(model_id).to(device)

c:\Users\USER\anaconda3\envs\aivle_pytorch\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProces

### (2) 모델 사용

* 이미지 예측 함수 생성
    * .convert("RGB") : 어떤 포맷의 이미지든 명시적으로 3채널 RGB로 변환되므로 안정적으로 처리 가능.

In [4]:
def predict_single_image(image_path: str):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model1(**inputs).logits
    pred = int(logits.argmax(-1))
    pred_label = model1.config.id2label.get(pred, str(pred))
    print(f"{image_path} -> {pred_label}")

In [ ]:
from google.colab import files
uploaded = files.upload() # 로컬 PC에서 a.png 선택

In [10]:
predict_single_image("./data/a.png")

./data/a.png -> recreational vehicle, RV, R.V.


In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 suni_bang1.jpg 선택

In [11]:
predict_single_image("./data/suni_bang1.jpg")

./data/suni_bang1.jpg -> Egyptian cat


## 3.파인튜닝 모델 : Fashion-mnist

### (1) 데이터 준비

In [12]:
# 케라스 데이터셋으로 부터 fashion_mnist 불러오기
(_, _), (x_val, y_val) = fashion_mnist.load_data()

In [13]:
# 타깃 클래스
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot" ]

### (2) 모델 다운로드
* 허깅페이스에 올라온 Fashion-MNIST 파인튜닝된 ViT 모델 : Kankanaghosh/vit-fashion-mnist
* 모델에 맞는 전처리기(processor)를 불러옴. (리사이즈, 정규화 등 자동 처리)
* 사전 학습된 ViT 분류 모델을 다운로드해서 GPU/CPU(device)에 올림

In [14]:
model_id = "Kankanaghosh/vit-fashion-mnist"
processor = AutoImageProcessor.from_pretrained(model_id)
model = ViTForImageClassification.from_pretrained(model_id).to(device)

c:\Users\USER\anaconda3\envs\aivle_pytorch\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--Kankanaghosh--vit-fashion-mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this mod

### (3) 검증평가

In [16]:
BATCH = 128   # 평가 배치 크기를 128로 설정
n = len(x_val)    # 검증 이미지 개수(n) 확인
y_pred = np.zeros(n, dtype=int)   # 예측 결과(y_pred)를 저장할 배열을 0으로 초기화

model.eval()   # 모델을 평가 모드로 전환
with torch.no_grad():    # 평가 시에는 학습(gradient 계산)이 필요 없으므로 메모리 절약을 위해 사용
    for i in tqdm(range(0, n, BATCH)):  # 0 ~ n까지 128단위로 반복
        j = min(i + BATCH, n)        # 마지막 배치에서 n을 넘지 않도록 범위 제한

        # Fashion-MNIST는 1채널 → ViT는 3채널 기대 → RGB 변환
        # 해당 배치 이미지를 PIL 이미지로 변환 후 리스트로 저장
        imgs = [Image.fromarray(x_val[k]).convert("RGB") for k in range(i, j)]

        # 이미지 전처리
        inputs = processor(images=imgs, return_tensors="pt").to(device)

        # 예측, 로짓값에 argmax 적용
        logits = model(**inputs).logits
        pred = logits.argmax(dim=-1).cpu().numpy()

        # 해당 배치 구간(i~j)에 예측값을 채워 넣음.
        y_pred[i:j] = pred


100%|██████████| 79/79 [00:58<00:00,  1.34it/s]


In [17]:
print(confusion_matrix(y_val, y_pred))
print('-'*100)
print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

[[869   0  21   8   3   1  95   0   3   0]
 [  0 990   2   3   3   0   1   0   1   0]
 [ 14   1 918   9  32   0  26   0   0   0]
 [  4   1   9 925  15   0  46   0   0   0]
 [  1   0  16   9 947   0  27   0   0   0]
 [  0   0   0   0   0 994   0   6   0   0]
 [ 48   0  44  13  44   0 848   0   3   0]
 [  0   0   0   0   0   2   0 984   0  14]
 [  0   0   1   0   1   0   3   0 995   0]
 [  0   0   0   0   0   7   0  25   0 968]]
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

 T-shirt/top     0.9284    0.8690    0.8977      1000
     Trouser     0.9980    0.9900    0.9940      1000
    Pullover     0.9080    0.9180    0.9130      1000
       Dress     0.9566    0.9250    0.9405      1000
        Coat     0.9062    0.9470    0.9262      1000
      Sandal     0.9900    0.9940    0.9920      1000
       Shirt     0.8107    0.8480    0.8289      1000
     Sneaker     0.9695    0.9840  

## 4.실습 : 파인튜닝 모델 : CIFAR10
* cifar10 데이터로 파인 튜닝된 모델에 대해 검증해 봅니다.

### (1) 데이터 준비

In [7]:
_, (x_val, y_val) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


In [8]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

### (2) 모델 다운로드

In [5]:
# vit - cifar10 파인튜닝 모델
model_id = "aaraki/vit-base-patch16-224-in21k-finetuned-cifar10"
processor = AutoImageProcessor.from_pretrained(model_id)
model = ViTForImageClassification.from_pretrained(model_id).to(device)

c:\Users\USER\anaconda3\envs\aivle_pytorch\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--aaraki--vit-base-patch16-224-in21k-finetuned-cifar10. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Using a slow image processor as `use_fast` is unset and a slow processor wa

### (3) 검증평가

In [10]:
BATCH = 128   # 평가 배치 크기를 128로 설정
n = len(x_val)    # 검증 이미지 개수(n) 확인
y_pred = np.zeros(n, dtype=int)   # 예측 결과(y_pred)를 저장할 배열을 0으로 초기화

model.eval()   # 모델을 평가 모드로 전환
with torch.no_grad():    # 평가 시에는 학습(gradient 계산)이 필요 없으므로 메모리 절약을 위해 사용
    for i in tqdm(range(0, n, BATCH)):  # 0 ~ n까지 128단위로 반복
        j = min(i + BATCH, n)        # 마지막 배치에서 n을 넘지 않도록 범위 제한

        # Fashion-MNIST는 1채널 → ViT는 3채널 기대 → RGB 변환
        # 해당 배치 이미지를 PIL 이미지로 변환 후 리스트로 저장
        imgs = [Image.fromarray(x_val[k]).convert("RGB") for k in range(i, j)]

        # 이미지 전처리
        inputs = processor(images=imgs, return_tensors="pt").to(device)

        # 예측, 로짓값에 argmax 적용
        logits = model(**inputs).logits
        pred = logits.argmax(dim=-1).cpu().numpy()

        # 해당 배치 구간(i~j)에 예측값을 채워 넣음.
        y_pred[i:j] = pred


100%|██████████| 79/79 [01:05<00:00,  1.21it/s]


In [11]:
print(confusion_matrix(y_val, y_pred))
print('-'*100)
print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

[[981   1   6   0   0   0   0   0  10   2]
 [  2 985   0   0   0   0   0   0   0  13]
 [  5   0 983   3   5   2   2   0   0   0]
 [  1   1   2 966   2  18   5   2   1   2]
 [  0   0   2   3 987   4   2   1   0   1]
 [  0   0   2  47   7 942   0   2   0   0]
 [  1   0   3   1   0   0 995   0   0   0]
 [  0   0   2   3   6   2   0 987   0   0]
 [  7   2   0   0   0   0   0   0 990   1]
 [  1  23   0   0   0   0   0   0   4 972]]
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

    airplane     0.9830    0.9810    0.9820      1000
  automobile     0.9733    0.9850    0.9791      1000
        bird     0.9830    0.9830    0.9830      1000
         cat     0.9443    0.9660    0.9550      1000
        deer     0.9801    0.9870    0.9836      1000
         dog     0.9731    0.9420    0.9573      1000
        frog     0.9910    0.9950    0.9930      1000
       horse     0.9950    0.9870  